# Temperatura promedio

In [30]:

#############
## Funciona #
#############
"""
Descarga datos HYRAS-DE-TAS (temperatura media diaria, 1 km, DWD-CDC),
recorta al área de los estados alemanes de interés y calcula el
promedio espacial diario para toda la región.

Fuente de datos:
https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/air_temperature_mean/

Requisitos (instalar antes de correr):
    pip install xarray rioxarray netCDF4 geopandas requests beautifulsoup4 pandas tqdm
"""

import re
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
import rioxarray  # noqa: F401  (habilita el accessor .rio en xarray)
import xarray as xr
from bs4 import BeautifulSoup
from tqdm import tqdm

# ------------------------------------------------------------------
# CONFIGURACIÓN — ajusta estos valores a tu caso
# ------------------------------------------------------------------
BASE_URL = (
    "https://opendata.dwd.de/climate_environment/CDC/"
    "grids_germany/daily/hyras_de/air_temperature_mean/"
)
AÑO_INICIO = 1996
AÑO_FIN = 2025
CARPETA_DESCARGA = Path("hyras_nc")

GPKG_ESTADOS = "States.gpkg"          # tu capa exportada desde QGIS (EPSG:4326)
CAMPO_NOMBRE_ESTADO = "NUTS_NAME"              # <-- ajusta al campo real de tu capa
ESTADOS_INTERES = ["Brandenburg"]  # <-- ajusta los nombres reales

SALIDA_CSV = "Brandenburg_MeanTemperature.csv"


def listar_archivos_nc(base_url: str) -> dict:
    """
    Parsea el índice HTML del DWD-CDC y devuelve {año: nombre_archivo},
    eligiendo automáticamente la versión más reciente disponible por año
    (ej. prioriza v6-1 sobre v6-0 si ambas existen).
    """
    resp = requests.get(base_url, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    patron = re.compile(r"tas_hyras_\d+_(\d{4})_v(\d+)-(\d+)_de\.nc$")
    candidatos = {}  # año -> (version_tuple, nombre_archivo)

    for link in soup.find_all("a"):
        href = link.get("href", "")
        m = patron.search(href)
        if not m:
            continue
        año, v_mayor, v_menor = int(m.group(1)), int(m.group(2)), int(m.group(3))
        version = (v_mayor, v_menor)
        if año not in candidatos or version > candidatos[año][0]:
            candidatos[año] = (version, href)

    return {año: nombre for año, (version, nombre) in candidatos.items()}


def descargar_archivo(base_url: str, nombre_archivo: str, carpeta_destino: Path) -> Path:
    """Descarga un .nc si no existe ya localmente (evita re-descargar)."""
    carpeta_destino.mkdir(parents=True, exist_ok=True)
    destino = carpeta_destino / nombre_archivo
    if destino.exists():
        return destino

    url = base_url + nombre_archivo
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
    return destino


def main():
    # 1. Localizar y descargar los archivos .nc necesarios
    print("Consultando índice de archivos disponibles en el DWD-CDC...")
    disponibles = listar_archivos_nc(BASE_URL)

    años_objetivo = [a for a in range(AÑO_INICIO, AÑO_FIN + 1) if a in disponibles]
    faltantes = [a for a in range(AÑO_INICIO, AÑO_FIN + 1) if a not in disponibles]
    if faltantes:
        print(f"Aviso: no se encontraron archivos para los años: {faltantes}")

    print(f"Descargando {len(años_objetivo)} archivos (se omiten los ya descargados)...")
    rutas_locales = []
    for año in tqdm(años_objetivo):
        ruta = descargar_archivo(BASE_URL, disponibles[año], CARPETA_DESCARGA)
        rutas_locales.append(ruta)

    # 2. Abrir todos los NetCDF como un solo dataset
    print("Abriendo archivos NetCDF...")
    ds = xr.open_mfdataset(
        rutas_locales,
        combine="by_coords"
    )[["tas"]]

    # HYRAS usa ETRS89 / LAEA Europe (EPSG:3035). La detección automática de
    # rioxarray a veces no interpreta bien la variable grid_mapping de estos
    # archivos, así que lo fijamos explícitamente si no se detectó solo.
    CRS_HYRAS = "EPSG:3035"
    if ds.rio.crs is None:
        print(f"CRS no detectado automáticamente; asignando {CRS_HYRAS} (confirmado para HYRAS-DE).")
        ds = ds.rio.write_crs(CRS_HYRAS, inplace=False)
    print(f"CRS usado para HYRAS: {ds.rio.crs}")

    # 3. Cargar y filtrar la capa de estados (exportada desde tu proyecto QGIS)
    print("Cargando capa de estados...")
    estados = gpd.read_file(GPKG_ESTADOS)
    print(f"Campos disponibles: {list(estados.columns)}")
    print(f"Valores en '{CAMPO_NOMBRE_ESTADO}': {sorted(estados[CAMPO_NOMBRE_ESTADO].unique())}")

    tres_estados = estados[estados[CAMPO_NOMBRE_ESTADO].isin(ESTADOS_INTERES)]
    if tres_estados.empty:
        raise ValueError(
            f"Ningún estado coincide con {ESTADOS_INTERES} en el campo "
            f"'{CAMPO_NOMBRE_ESTADO}'. Revisa la lista impresa arriba y ajusta "
            "ESTADOS_INTERES con los nombres exactos."
        )

    # Reproyectar la capa vectorial al CRS del raster HYRAS
    tres_estados = tres_estados.to_crs(ds.rio.crs)

    # 4. Recortar el dataset a la geometría combinada de los tres estados
    print("Recortando a la región de interés...")
    geometria_union = tres_estados.dissolve().geometry
    ds_recortado = ds.rio.clip(geometria_union, tres_estados.crs)

    # 5. Calcular el promedio espacial diario
    print("Calculando promedio diario...")
    dims_espaciales = [d for d in ds_recortado["tas"].dims if d != "time"]
    promedio_diario = ds_recortado["tas"].mean(dim=dims_espaciales, skipna=True)

    # 6. Exportar a CSV
    df = promedio_diario.to_dataframe().reset_index()
    df = df.rename(columns={"tas": "temp_media_c"})[["time", "temp_media_c"]]

    # Aviso: confirma en el PDF de descripción si 'tas' viene en °C o en Kelvin.
    # Si viene en Kelvin, descomenta la siguiente línea:
    # df["temp_media_c"] = df["temp_media_c"] - 273.15

    df.to_csv(SALIDA_CSV, index=False)
    print(f"Listo. Serie diaria guardada en: {SALIDA_CSV} ({len(df)} días)")


if __name__ == "__main__":
    main()


Consultando índice de archivos disponibles en el DWD-CDC...
Descargando 30 archivos (se omiten los ya descargados)...


100%|██████████| 30/30 [00:00<00:00, 3759.01it/s]

Abriendo archivos NetCDF...



C:\Users\amigo\AppData\Local\Temp\ipykernel_31024\4280367281.py:105: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


CRS no detectado automáticamente; asignando EPSG:3035 (confirmado para HYRAS-DE).
CRS usado para HYRAS: EPSG:3035
Cargando capa de estados...
Campos disponibles: ['id', 'OBJID', 'BEGINN', 'GF', 'NUTS_LEVEL', 'NUTS_CODE', 'NUTS_NAME', 'ThreeRegions', 'TEST', 'geometry']
Valores en 'NUTS_NAME': ['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen']
Recortando a la región de interés...
Calculando promedio diario...
Listo. Serie diaria guardada en: Brandenburg_MeanTemperature.csv (10958 días)


# Temperatura máxima

In [11]:

#############
## Funciona #
#############
"""
Descarga datos HYRAS-DE-TAS (temperatura media diaria, 1 km, DWD-CDC),
recorta al área de los estados alemanes de interés y calcula el
promedio espacial diario para toda la región.

Fuente de datos:
https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/air_temperature_mean/

Requisitos (instalar antes de correr):
    pip install xarray rioxarray netCDF4 geopandas requests beautifulsoup4 pandas tqdm
"""

import re
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
import rioxarray  # noqa: F401  (habilita el accessor .rio en xarray)
import xarray as xr
from bs4 import BeautifulSoup
from tqdm import tqdm

# ------------------------------------------------------------------
# CONFIGURACIÓN — ajusta estos valores a tu caso
# ------------------------------------------------------------------
BASE_URL = (
    "https://opendata.dwd.de/climate_environment/CDC/"
    "grids_germany/daily/hyras_de/air_temperature_max/"
)
AÑO_INICIO = 1996
AÑO_FIN = 2025
CARPETA_DESCARGA = Path("Max_temperature_hyras_nc")

GPKG_ESTADOS = "States.gpkg"          # tu capa exportada desde QGIS (EPSG:4326)
CAMPO_NOMBRE_ESTADO = "NUTS_NAME"              # <-- ajusta al campo real de tu capa
ESTADOS_INTERES = ["Brandenburg"]  # <-- ajusta los nombres reales

SALIDA_CSV = "Brandenburg_MaxTemperature.csv"


def listar_archivos_nc(base_url: str) -> dict:
    """
    Parsea el índice HTML del DWD-CDC y devuelve {año: nombre_archivo},
    eligiendo automáticamente la versión más reciente disponible por año
    (ej. prioriza v6-1 sobre v6-0 si ambas existen).
    """
    resp = requests.get(base_url, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    patron = re.compile(r"tasmax_hyras_\d+_(\d{4})_v(\d+)-(\d+)_de\.nc$")
    candidatos = {}  # año -> (version_tuple, nombre_archivo)

    for link in soup.find_all("a"):
        href = link.get("href", "")
        m = patron.search(href)
        if not m:
            continue
        año, v_mayor, v_menor = int(m.group(1)), int(m.group(2)), int(m.group(3))
        version = (v_mayor, v_menor)
        if año not in candidatos or version > candidatos[año][0]:
            candidatos[año] = (version, href)

    return {año: nombre for año, (version, nombre) in candidatos.items()}


def descargar_archivo(base_url: str, nombre_archivo: str, carpeta_destino: Path) -> Path:
    """Descarga un .nc si no existe ya localmente (evita re-descargar)."""
    carpeta_destino.mkdir(parents=True, exist_ok=True)
    destino = carpeta_destino / nombre_archivo
    if destino.exists():
        return destino

    url = base_url + nombre_archivo
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
    return destino


def main():
    # 1. Localizar y descargar los archivos .nc necesarios
    print("Consultando índice de archivos disponibles en el DWD-CDC...")
    disponibles = listar_archivos_nc(BASE_URL)

    años_objetivo = [a for a in range(AÑO_INICIO, AÑO_FIN + 1) if a in disponibles]
    faltantes = [a for a in range(AÑO_INICIO, AÑO_FIN + 1) if a not in disponibles]
    if faltantes:
        print(f"Aviso: no se encontraron archivos para los años: {faltantes}")

    print(f"Descargando {len(años_objetivo)} archivos (se omiten los ya descargados)...")
    rutas_locales = []
    for año in tqdm(años_objetivo):
        ruta = descargar_archivo(BASE_URL, disponibles[año], CARPETA_DESCARGA)
        rutas_locales.append(ruta)

    # 2. Abrir todos los NetCDF como un solo dataset
    print("Abriendo archivos NetCDF...")
    ds = xr.open_mfdataset(
        rutas_locales,
        combine="by_coords"
    )[["tasmax"]]

    # HYRAS usa ETRS89 / LAEA Europe (EPSG:3035). La detección automática de
    # rioxarray a veces no interpreta bien la variable grid_mapping de estos
    # archivos, así que lo fijamos explícitamente si no se detectó solo.
    CRS_HYRAS = "EPSG:3035"
    if ds.rio.crs is None:
        print(f"CRS no detectado automáticamente; asignando {CRS_HYRAS} (confirmado para HYRAS-DE).")
        ds = ds.rio.write_crs(CRS_HYRAS, inplace=False)
    print(f"CRS usado para HYRAS: {ds.rio.crs}")

    # 3. Cargar y filtrar la capa de estados (exportada desde tu proyecto QGIS)
    print("Cargando capa de estados...")
    estados = gpd.read_file(GPKG_ESTADOS)
    print(f"Campos disponibles: {list(estados.columns)}")
    print(f"Valores en '{CAMPO_NOMBRE_ESTADO}': {sorted(estados[CAMPO_NOMBRE_ESTADO].unique())}")

    tres_estados = estados[estados[CAMPO_NOMBRE_ESTADO].isin(ESTADOS_INTERES)]
    if tres_estados.empty:
        raise ValueError(
            f"Ningún estado coincide con {ESTADOS_INTERES} en el campo "
            f"'{CAMPO_NOMBRE_ESTADO}'. Revisa la lista impresa arriba y ajusta "
            "ESTADOS_INTERES con los nombres exactos."
        )

    # Reproyectar la capa vectorial al CRS del raster HYRAS
    tres_estados = tres_estados.to_crs(ds.rio.crs)

    # 4. Recortar el dataset a la geometría combinada de los tres estados
    print("Recortando a la región de interés...")
    geometria_union = tres_estados.dissolve().geometry
    ds_recortado = ds.rio.clip(geometria_union, tres_estados.crs)

    # 5. Calcular el promedio espacial diario
    print("Calculando promedio diario...")
    dims_espaciales = [d for d in ds_recortado["tasmax"].dims if d != "time"]
    promedio_diario = ds_recortado["tasmax"].mean(dim=dims_espaciales, skipna=True)

    # 6. Exportar a CSV
    df = promedio_diario.to_dataframe().reset_index()
    df = df.rename(columns={"tasmax": "temp_max_c"})[["time", "temp_max_c"]]

    # Aviso: confirma en el PDF de descripción si 'tas' viene en °C o en Kelvin.
    # Si viene en Kelvin, descomenta la siguiente línea:
    # df["temp_media_c"] = df["temp_media_c"] - 273.15

    df.to_csv(SALIDA_CSV, index=False)
    print(f"Listo. Serie diaria guardada en: {SALIDA_CSV} ({len(df)} días)")


if __name__ == "__main__":
    main()


Consultando índice de archivos disponibles en el DWD-CDC...
Descargando 30 archivos (se omiten los ya descargados)...


100%|██████████| 30/30 [00:00<00:00, 3147.22it/s]

Abriendo archivos NetCDF...



C:\Users\amigo\AppData\Local\Temp\ipykernel_15384\3912347423.py:105: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


CRS no detectado automáticamente; asignando EPSG:3035 (confirmado para HYRAS-DE).
CRS usado para HYRAS: EPSG:3035
Cargando capa de estados...
Campos disponibles: ['id', 'OBJID', 'BEGINN', 'GF', 'NUTS_LEVEL', 'NUTS_CODE', 'NUTS_NAME', 'ThreeRegions', 'TEST', 'geometry']
Valores en 'NUTS_NAME': ['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen']
Recortando a la región de interés...
Calculando promedio diario...


KeyboardInterrupt: 

# Temperatura mínima

In [12]:
#############
## Funciona #
#############
"""
Descarga datos HYRAS-DE-TAS (temperatura media diaria, 1 km, DWD-CDC),
recorta al área de los estados alemanes de interés y calcula el
promedio espacial diario para toda la región.

Fuente de datos:
https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/air_temperature_mean/

Requisitos (instalar antes de correr):
    pip install xarray rioxarray netCDF4 geopandas requests beautifulsoup4 pandas tqdm
"""

import re
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
import rioxarray  # noqa: F401  (habilita el accessor .rio en xarray)
import xarray as xr
from bs4 import BeautifulSoup
from tqdm import tqdm

# ------------------------------------------------------------------
# CONFIGURACIÓN — ajusta estos valores a tu caso
# ------------------------------------------------------------------
BASE_URL = (
    "https://opendata.dwd.de/climate_environment/CDC/"
    "grids_germany/daily/hyras_de/air_temperature_min/"
)
AÑO_INICIO = 1996
AÑO_FIN = 2025
CARPETA_DESCARGA = Path("Min_temperature_hyras_nc")

GPKG_ESTADOS = "States.gpkg"          # tu capa exportada desde QGIS (EPSG:4326)
CAMPO_NOMBRE_ESTADO = "NUTS_NAME"              # <-- ajusta al campo real de tu capa
ESTADOS_INTERES = ["Brandenburg"]  # <-- ajusta los nombres reales

SALIDA_CSV = "Brandenburg_MinTemperature.csv"


def listar_archivos_nc(base_url: str) -> dict:
    """
    Parsea el índice HTML del DWD-CDC y devuelve {año: nombre_archivo},
    eligiendo automáticamente la versión más reciente disponible por año
    (ej. prioriza v6-1 sobre v6-0 si ambas existen).
    """
    resp = requests.get(base_url, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    patron = re.compile(r"tasmin_hyras_\d+_(\d{4})_v(\d+)-(\d+)_de\.nc$")
    candidatos = {}  # año -> (version_tuple, nombre_archivo)

    for link in soup.find_all("a"):
        href = link.get("href", "")
        m = patron.search(href)
        if not m:
            continue
        año, v_mayor, v_menor = int(m.group(1)), int(m.group(2)), int(m.group(3))
        version = (v_mayor, v_menor)
        if año not in candidatos or version > candidatos[año][0]:
            candidatos[año] = (version, href)

    return {año: nombre for año, (version, nombre) in candidatos.items()}


def descargar_archivo(base_url: str, nombre_archivo: str, carpeta_destino: Path) -> Path:
    """Descarga un .nc si no existe ya localmente (evita re-descargar)."""
    carpeta_destino.mkdir(parents=True, exist_ok=True)
    destino = carpeta_destino / nombre_archivo
    if destino.exists():
        return destino

    url = base_url + nombre_archivo
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
    return destino


def main():
    # 1. Localizar y descargar los archivos .nc necesarios
    print("Consultando índice de archivos disponibles en el DWD-CDC...")
    disponibles = listar_archivos_nc(BASE_URL)

    años_objetivo = [a for a in range(AÑO_INICIO, AÑO_FIN + 1) if a in disponibles]
    faltantes = [a for a in range(AÑO_INICIO, AÑO_FIN + 1) if a not in disponibles]
    if faltantes:
        print(f"Aviso: no se encontraron archivos para los años: {faltantes}")

    print(f"Descargando {len(años_objetivo)} archivos (se omiten los ya descargados)...")
    rutas_locales = []
    for año in tqdm(años_objetivo):
        ruta = descargar_archivo(BASE_URL, disponibles[año], CARPETA_DESCARGA)
        rutas_locales.append(ruta)

    # 2. Abrir todos los NetCDF como un solo dataset
    print("Abriendo archivos NetCDF...")
    ds = xr.open_mfdataset(
        rutas_locales,
        combine="by_coords"
    )[["tasmin"]]

    # HYRAS usa ETRS89 / LAEA Europe (EPSG:3035). La detección automática de
    # rioxarray a veces no interpreta bien la variable grid_mapping de estos
    # archivos, así que lo fijamos explícitamente si no se detectó solo.
    CRS_HYRAS = "EPSG:3035"
    if ds.rio.crs is None:
        print(f"CRS no detectado automáticamente; asignando {CRS_HYRAS} (confirmado para HYRAS-DE).")
        ds = ds.rio.write_crs(CRS_HYRAS, inplace=False)
    print(f"CRS usado para HYRAS: {ds.rio.crs}")

    # 3. Cargar y filtrar la capa de estados (exportada desde tu proyecto QGIS)
    print("Cargando capa de estados...")
    estados = gpd.read_file(GPKG_ESTADOS)
    print(f"Campos disponibles: {list(estados.columns)}")
    print(f"Valores en '{CAMPO_NOMBRE_ESTADO}': {sorted(estados[CAMPO_NOMBRE_ESTADO].unique())}")

    tres_estados = estados[estados[CAMPO_NOMBRE_ESTADO].isin(ESTADOS_INTERES)]
    if tres_estados.empty:
        raise ValueError(
            f"Ningún estado coincide con {ESTADOS_INTERES} en el campo "
            f"'{CAMPO_NOMBRE_ESTADO}'. Revisa la lista impresa arriba y ajusta "
            "ESTADOS_INTERES con los nombres exactos."
        )

    # Reproyectar la capa vectorial al CRS del raster HYRAS
    tres_estados = tres_estados.to_crs(ds.rio.crs)

    # 4. Recortar el dataset a la geometría combinada de los tres estados
    print("Recortando a la región de interés...")
    geometria_union = tres_estados.dissolve().geometry
    ds_recortado = ds.rio.clip(geometria_union, tres_estados.crs)

    # 5. Calcular el promedio espacial diario
    print("Calculando promedio diario...")
    dims_espaciales = [d for d in ds_recortado["tasmin"].dims if d != "time"]
    promedio_diario = ds_recortado["tasmin"].mean(dim=dims_espaciales, skipna=True)

    # 6. Exportar a CSV
    df = promedio_diario.to_dataframe().reset_index()
    df = df.rename(columns={"tasmin": "temp_min_c"})[["time", "temp_min_c"]]

    # Aviso: confirma en el PDF de descripción si 'tas' viene en °C o en Kelvin.
    # Si viene en Kelvin, descomenta la siguiente línea:
    # df["temp_media_c"] = df["temp_media_c"] - 273.15

    df.to_csv(SALIDA_CSV, index=False)
    print(f"Listo. Serie diaria guardada en: {SALIDA_CSV} ({len(df)} días)")


if __name__ == "__main__":
    main()


Consultando índice de archivos disponibles en el DWD-CDC...
Descargando 30 archivos (se omiten los ya descargados)...


100%|██████████| 30/30 [00:00<00:00, 3958.76it/s]

Abriendo archivos NetCDF...



C:\Users\amigo\AppData\Local\Temp\ipykernel_15384\1599186789.py:105: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


CRS no detectado automáticamente; asignando EPSG:3035 (confirmado para HYRAS-DE).
CRS usado para HYRAS: EPSG:3035
Cargando capa de estados...
Campos disponibles: ['id', 'OBJID', 'BEGINN', 'GF', 'NUTS_LEVEL', 'NUTS_CODE', 'NUTS_NAME', 'ThreeRegions', 'TEST', 'geometry']
Valores en 'NUTS_NAME': ['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen']
Recortando a la región de interés...
Calculando promedio diario...
Listo. Serie diaria guardada en: Brandenburg_MinTemperature.csv (10958 días)


# Humedad relativa

In [15]:
#############
## Funciona #
#############
"""
Descarga datos HYRAS-DE-TAS (temperatura media diaria, 1 km, DWD-CDC),
recorta al área de los estados alemanes de interés y calcula el
promedio espacial diario para toda la región.

Fuente de datos:
https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/air_temperature_mean/

Requisitos (instalar antes de correr):
    pip install xarray rioxarray netCDF4 geopandas requests beautifulsoup4 pandas tqdm
"""

import re
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
import rioxarray  # noqa: F401  (habilita el accessor .rio en xarray)
import xarray as xr
from bs4 import BeautifulSoup
from tqdm import tqdm

# ------------------------------------------------------------------
# CONFIGURACIÓN — ajusta estos valores a tu caso
# ------------------------------------------------------------------
BASE_URL = (
    "https://opendata.dwd.de/climate_environment/CDC/"
    "grids_germany/daily/hyras_de/humidity/"
)
AÑO_INICIO = 1996
AÑO_FIN = 2025
CARPETA_DESCARGA = Path("Humidity_hyras_nc")

GPKG_ESTADOS = "States.gpkg"          # tu capa exportada desde QGIS (EPSG:4326)
CAMPO_NOMBRE_ESTADO = "NUTS_NAME"              # <-- ajusta al campo real de tu capa
ESTADOS_INTERES = ["Bayern"]  # <-- ajusta los nombres reales

SALIDA_CSV = "Bayern_Humidity.csv"


def listar_archivos_nc(base_url: str) -> dict:
    """
    Parsea el índice HTML del DWD-CDC y devuelve {año: nombre_archivo},
    eligiendo automáticamente la versión más reciente disponible por año
    (ej. prioriza v6-1 sobre v6-0 si ambas existen).
    """
    resp = requests.get(base_url, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    patron = re.compile(r"hurs_hyras_\d+_(\d{4})_v(\d+)-(\d+)_de\.nc$")
    candidatos = {}  # año -> (version_tuple, nombre_archivo)

    for link in soup.find_all("a"):
        href = link.get("href", "")
        m = patron.search(href)
        if not m:
            continue
        año, v_mayor, v_menor = int(m.group(1)), int(m.group(2)), int(m.group(3))
        version = (v_mayor, v_menor)
        if año not in candidatos or version > candidatos[año][0]:
            candidatos[año] = (version, href)

    return {año: nombre for año, (version, nombre) in candidatos.items()}


def descargar_archivo(base_url: str, nombre_archivo: str, carpeta_destino: Path) -> Path:
    """Descarga un .nc si no existe ya localmente (evita re-descargar)."""
    carpeta_destino.mkdir(parents=True, exist_ok=True)
    destino = carpeta_destino / nombre_archivo
    if destino.exists():
        return destino

    url = base_url + nombre_archivo
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
    return destino


def main():
    # 1. Localizar y descargar los archivos .nc necesarios
    print("Consultando índice de archivos disponibles en el DWD-CDC...")
    disponibles = listar_archivos_nc(BASE_URL)

    años_objetivo = [a for a in range(AÑO_INICIO, AÑO_FIN + 1) if a in disponibles]
    faltantes = [a for a in range(AÑO_INICIO, AÑO_FIN + 1) if a not in disponibles]
    if faltantes:
        print(f"Aviso: no se encontraron archivos para los años: {faltantes}")

    print(f"Descargando {len(años_objetivo)} archivos (se omiten los ya descargados)...")
    rutas_locales = []
    for año in tqdm(años_objetivo):
        ruta = descargar_archivo(BASE_URL, disponibles[año], CARPETA_DESCARGA)
        rutas_locales.append(ruta)

    # 2. Abrir todos los NetCDF como un solo dataset
    print("Abriendo archivos NetCDF...")
    ds = xr.open_mfdataset(
        rutas_locales,
        combine="by_coords"
    )[["hurs"]]

    # HYRAS usa ETRS89 / LAEA Europe (EPSG:3035). La detección automática de
    # rioxarray a veces no interpreta bien la variable grid_mapping de estos
    # archivos, así que lo fijamos explícitamente si no se detectó solo.
    CRS_HYRAS = "EPSG:3035"
    if ds.rio.crs is None:
        print(f"CRS no detectado automáticamente; asignando {CRS_HYRAS} (confirmado para HYRAS-DE).")
        ds = ds.rio.write_crs(CRS_HYRAS, inplace=False)
    print(f"CRS usado para HYRAS: {ds.rio.crs}")

    # 3. Cargar y filtrar la capa de estados (exportada desde tu proyecto QGIS)
    print("Cargando capa de estados...")
    estados = gpd.read_file(GPKG_ESTADOS)
    print(f"Campos disponibles: {list(estados.columns)}")
    print(f"Valores en '{CAMPO_NOMBRE_ESTADO}': {sorted(estados[CAMPO_NOMBRE_ESTADO].unique())}")

    tres_estados = estados[estados[CAMPO_NOMBRE_ESTADO].isin(ESTADOS_INTERES)]
    if tres_estados.empty:
        raise ValueError(
            f"Ningún estado coincide con {ESTADOS_INTERES} en el campo "
            f"'{CAMPO_NOMBRE_ESTADO}'. Revisa la lista impresa arriba y ajusta "
            "ESTADOS_INTERES con los nombres exactos."
        )

    # Reproyectar la capa vectorial al CRS del raster HYRAS
    tres_estados = tres_estados.to_crs(ds.rio.crs)

    # 4. Recortar el dataset a la geometría combinada de los tres estados
    print("Recortando a la región de interés...")
    geometria_union = tres_estados.dissolve().geometry
    ds_recortado = ds.rio.clip(geometria_union, tres_estados.crs)

    # 5. Calcular el promedio espacial diario
    print("Calculando promedio diario...")
    dims_espaciales = [d for d in ds_recortado["hurs"].dims if d != "time"]
    promedio_diario = ds_recortado["hurs"].mean(dim=dims_espaciales, skipna=True)

    # 6. Exportar a CSV
    df = promedio_diario.to_dataframe().reset_index()
    df = df.rename(columns={"hurs": "relative_humidity_%"})[["time", "relative_humidity_%"]]

    # Aviso: confirma en el PDF de descripción si 'tas' viene en °C o en Kelvin.
    # Si viene en Kelvin, descomenta la siguiente línea:
    # df["temp_media_c"] = df["temp_media_c"] - 273.15

    df.to_csv(SALIDA_CSV, index=False)
    print(f"Listo. Serie diaria guardada en: {SALIDA_CSV} ({len(df)} días)")


if __name__ == "__main__":
    main()


Consultando índice de archivos disponibles en el DWD-CDC...
Descargando 30 archivos (se omiten los ya descargados)...


100%|██████████| 30/30 [00:00<00:00, 2141.48it/s]

Abriendo archivos NetCDF...



C:\Users\amigo\AppData\Local\Temp\ipykernel_15384\4120962272.py:105: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


CRS no detectado automáticamente; asignando EPSG:3035 (confirmado para HYRAS-DE).
CRS usado para HYRAS: EPSG:3035
Cargando capa de estados...
Campos disponibles: ['id', 'OBJID', 'BEGINN', 'GF', 'NUTS_LEVEL', 'NUTS_CODE', 'NUTS_NAME', 'ThreeRegions', 'TEST', 'geometry']
Valores en 'NUTS_NAME': ['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen']
Recortando a la región de interés...
Calculando promedio diario...
Listo. Serie diaria guardada en: Bayern_Humidity.csv (10958 días)


# Precipitación

In [18]:
#############
## Funciona #
#############
"""
Descarga datos HYRAS-DE-TAS (temperatura media diaria, 1 km, DWD-CDC),
recorta al área de los estados alemanes de interés y calcula el
promedio espacial diario para toda la región.

Fuente de datos:
https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/air_temperature_mean/

Requisitos (instalar antes de correr):
    pip install xarray rioxarray netCDF4 geopandas requests beautifulsoup4 pandas tqdm
"""

import re
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
import rioxarray  # noqa: F401  (habilita el accessor .rio en xarray)
import xarray as xr
from bs4 import BeautifulSoup
from tqdm import tqdm

# ------------------------------------------------------------------
# CONFIGURACIÓN — ajusta estos valores a tu caso
# ------------------------------------------------------------------
BASE_URL = (
    "https://opendata.dwd.de/climate_environment/CDC/"
    "grids_germany/daily/hyras_de/precipitation/"
)
AÑO_INICIO = 1996
AÑO_FIN = 2025
CARPETA_DESCARGA = Path("Precipitation_hyras_nc")

GPKG_ESTADOS = "States.gpkg"          # tu capa exportada desde QGIS (EPSG:4326)
CAMPO_NOMBRE_ESTADO = "NUTS_NAME"              # <-- ajusta al campo real de tu capa
ESTADOS_INTERES = ["Brandenburg"]  # <-- ajusta los nombres reales

SALIDA_CSV = "Brandenburg_Precipitation.csv"


def listar_archivos_nc(base_url: str) -> dict:
    """
    Parsea el índice HTML del DWD-CDC y devuelve {año: nombre_archivo},
    eligiendo automáticamente la versión más reciente disponible por año
    (ej. prioriza v6-1 sobre v6-0 si ambas existen).
    """
    resp = requests.get(base_url, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    patron = re.compile(r"pr_hyras_\d+_(\d{4})_v(\d+)-(\d+)_de\.nc$")
    candidatos = {}  # año -> (version_tuple, nombre_archivo)

    for link in soup.find_all("a"):
        href = link.get("href", "")
        m = patron.search(href)
        if not m:
            continue
        año, v_mayor, v_menor = int(m.group(1)), int(m.group(2)), int(m.group(3))
        version = (v_mayor, v_menor)
        if año not in candidatos or version > candidatos[año][0]:
            candidatos[año] = (version, href)

    return {año: nombre for año, (version, nombre) in candidatos.items()}


def descargar_archivo(base_url: str, nombre_archivo: str, carpeta_destino: Path) -> Path:
    """Descarga un .nc si no existe ya localmente (evita re-descargar)."""
    carpeta_destino.mkdir(parents=True, exist_ok=True)
    destino = carpeta_destino / nombre_archivo
    if destino.exists():
        return destino

    url = base_url + nombre_archivo
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
    return destino


def main():
    # 1. Localizar y descargar los archivos .nc necesarios
    print("Consultando índice de archivos disponibles en el DWD-CDC...")
    disponibles = listar_archivos_nc(BASE_URL)

    años_objetivo = [a for a in range(AÑO_INICIO, AÑO_FIN + 1) if a in disponibles]
    faltantes = [a for a in range(AÑO_INICIO, AÑO_FIN + 1) if a not in disponibles]
    if faltantes:
        print(f"Aviso: no se encontraron archivos para los años: {faltantes}")

    print(f"Descargando {len(años_objetivo)} archivos (se omiten los ya descargados)...")
    rutas_locales = []
    for año in tqdm(años_objetivo):
        ruta = descargar_archivo(BASE_URL, disponibles[año], CARPETA_DESCARGA)
        rutas_locales.append(ruta)

    # 2. Abrir todos los NetCDF como un solo dataset
    print("Abriendo archivos NetCDF...")
    ds = xr.open_mfdataset(
        rutas_locales,
        combine="by_coords"
    )[["pr"]]

    # HYRAS usa ETRS89 / LAEA Europe (EPSG:3035). La detección automática de
    # rioxarray a veces no interpreta bien la variable grid_mapping de estos
    # archivos, así que lo fijamos explícitamente si no se detectó solo.
    CRS_HYRAS = "EPSG:3035"
    if ds.rio.crs is None:
        print(f"CRS no detectado automáticamente; asignando {CRS_HYRAS} (confirmado para HYRAS-DE).")
        ds = ds.rio.write_crs(CRS_HYRAS, inplace=False)
    print(f"CRS usado para HYRAS: {ds.rio.crs}")

    # 3. Cargar y filtrar la capa de estados (exportada desde tu proyecto QGIS)
    print("Cargando capa de estados...")
    estados = gpd.read_file(GPKG_ESTADOS)
    print(f"Campos disponibles: {list(estados.columns)}")
    print(f"Valores en '{CAMPO_NOMBRE_ESTADO}': {sorted(estados[CAMPO_NOMBRE_ESTADO].unique())}")

    tres_estados = estados[estados[CAMPO_NOMBRE_ESTADO].isin(ESTADOS_INTERES)]
    if tres_estados.empty:
        raise ValueError(
            f"Ningún estado coincide con {ESTADOS_INTERES} en el campo "
            f"'{CAMPO_NOMBRE_ESTADO}'. Revisa la lista impresa arriba y ajusta "
            "ESTADOS_INTERES con los nombres exactos."
        )

    # Reproyectar la capa vectorial al CRS del raster HYRAS
    tres_estados = tres_estados.to_crs(ds.rio.crs)

    # 4. Recortar el dataset a la geometría combinada de los tres estados
    print("Recortando a la región de interés...")
    geometria_union = tres_estados.dissolve().geometry
    ds_recortado = ds.rio.clip(geometria_union, tres_estados.crs)

    # 5. Calcular el promedio espacial diario
    print("Calculando promedio diario...")
    dims_espaciales = [d for d in ds_recortado["pr"].dims if d != "time"]
    promedio_diario = ds_recortado["pr"].mean(dim=dims_espaciales, skipna=True)

    # 6. Exportar a CSV
    df = promedio_diario.to_dataframe().reset_index()
    df = df.rename(columns={"pr": "precipitation_mm"})[["time", "precipitation_mm"]]

    # Aviso: confirma en el PDF de descripción si 'tas' viene en °C o en Kelvin.
    # Si viene en Kelvin, descomenta la siguiente línea:
    # df["temp_media_c"] = df["temp_media_c"] - 273.15

    df.to_csv(SALIDA_CSV, index=False)
    print(f"Listo. Serie diaria guardada en: {SALIDA_CSV} ({len(df)} días)")


if __name__ == "__main__":
    main()


Consultando índice de archivos disponibles en el DWD-CDC...
Descargando 30 archivos (se omiten los ya descargados)...


100%|██████████| 30/30 [00:00<00:00, 2730.73it/s]

Abriendo archivos NetCDF...



C:\Users\amigo\AppData\Local\Temp\ipykernel_15384\3800247945.py:105: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


CRS no detectado automáticamente; asignando EPSG:3035 (confirmado para HYRAS-DE).
CRS usado para HYRAS: EPSG:3035
Cargando capa de estados...
Campos disponibles: ['id', 'OBJID', 'BEGINN', 'GF', 'NUTS_LEVEL', 'NUTS_CODE', 'NUTS_NAME', 'ThreeRegions', 'TEST', 'geometry']
Valores en 'NUTS_NAME': ['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen']
Recortando a la región de interés...
Calculando promedio diario...
Listo. Serie diaria guardada en: Brandenburg_Precipitation.csv (10958 días)


# Temperatura del suelo

In [5]:
# ==========================================================
# DWD Soil Temperature 5 cm - Sachsen
# Descarga, extracción temporal, recorte espacial y agregación
# ==========================================================

from pathlib import Path
import requests
import tarfile
import tempfile

import geopandas as gpd
import rasterio
from rasterio.mask import mask

import pandas as pd
import numpy as np


# ==========================================================
# CONFIGURACIÓN
# ==========================================================

YEARS = range(1996, 2026)

DWD_URL = (
    "https://opendata.dwd.de/"
    "climate_environment/CDC/grids_germany/daily/"
    "soil_temperature_5cm/"
)

DOWNLOAD_DIR = Path("DWD_soil_temperature_5cm")
DOWNLOAD_DIR.mkdir(exist_ok=True)

STATES_FILE = "States.gpkg"

OUTPUT_FILE = "2Brandenburg_soil_temperature_5cm.csv"


# ==========================================================
# 1. Cargar Sachsen
# ==========================================================

print("Leyendo Estados...")

states = gpd.read_file("States.gpkg")

sachsen = states[
    states["NUTS_NAME"] == "Brandenburg"
].copy()

print(sachsen.crs)


if len(sachsen) == 0:
    raise ValueError(
        "No se encontró Sachsen en NUTS_NAME"
    )


print(
    "CRS Sachsen:",
    sachsen.crs
)


# ==========================================================
# 2. Descargar archivos mensuales DWD
# ==========================================================

def download_month(year, month):

    filename = (
        "grids_germany_daily_soil_temperature_5cm_"
        f"{year}{month:02d}.tgz"
    )

    filepath = DOWNLOAD_DIR / filename


    if filepath.exists():
        return filepath


    url = DWD_URL + filename


    print("Descargando:", filename)


    r = requests.get(
        url,
        timeout=120
    )


    if r.status_code != 200:
        raise RuntimeError(
            f"No existe {url}"
        )


    filepath.write_bytes(
        r.content
    )


    return filepath



# ==========================================================
# 3. Procesar un archivo mensual
# ==========================================================

def process_month(tgz_file):

    records = []


    with tempfile.TemporaryDirectory() as tmp:


        # ----------------------------------------------
        # Extraer temporalmente
        # ----------------------------------------------

        with tarfile.open(tgz_file) as tar:
            tar.extractall(tmp)


        asc_files = sorted(
            Path(tmp).rglob("*.asc")
        )


        print(
            tgz_file.name,
            "->",
            len(asc_files),
            "días"
        )


        # ----------------------------------------------
        # Procesar cada día
        # ----------------------------------------------

        for asc in asc_files:


            # ejemplo:
            # soil_temperature_5cm_19910101.asc

            date_string = asc.stem[-8:]


            with rasterio.open(asc) as src:

                geom = (
                    sachsen
                    .to_crs("EPSG:31467")
                    .geometry
                )


                clipped, _ = mask(
                    src,
                    geom,
                    crop=True
                )


                values = clipped[0]


                values = values[
                    values != src.nodata
                ]


                mean_temp = values.mean()/10


                records.append(
                    {
                        "date":
                            pd.to_datetime(
                                date_string,
                                format="%Y%m%d"
                            ),

                        "state":
                            "Sachsen",

                        "soil_temp_5cm":
                            mean_temp
                    }
                )


    return records



# ==========================================================
# 4. Loop completo 1994-2025
# ==========================================================

all_records = []


for year in YEARS:

    for month in range(1,13):

        tgz = download_month(
            year,
            month
        )


        month_records = process_month(
            tgz
        )


        all_records.extend(
            month_records
        )


        print(
            "Registros acumulados:",
            len(all_records)
        )



# ==========================================================
# 5. Crear DataFrame final
# ==========================================================

df = pd.DataFrame(
    all_records
)


df = df.sort_values(
    "date"
)


# agregar día del año

df["DOY"] = (
    df["date"]
    .dt.dayofyear
)


# ==========================================================
# 6. Guardar
# ==========================================================


df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n================================")
print("Proceso terminado")
print("Archivo:", OUTPUT_FILE)
print("Filas:", len(df))
print(df.head())
print("================================")

Leyendo Estados...
EPSG:4326
CRS Sachsen: EPSG:4326
grids_germany_daily_soil_temperature_5cm_199601.tgz -> 31 días
Registros acumulados: 31
grids_germany_daily_soil_temperature_5cm_199602.tgz -> 29 días
Registros acumulados: 60
grids_germany_daily_soil_temperature_5cm_199603.tgz -> 31 días
Registros acumulados: 91
grids_germany_daily_soil_temperature_5cm_199604.tgz -> 30 días
Registros acumulados: 121
grids_germany_daily_soil_temperature_5cm_199605.tgz -> 31 días
Registros acumulados: 152
grids_germany_daily_soil_temperature_5cm_199606.tgz -> 30 días
Registros acumulados: 182
grids_germany_daily_soil_temperature_5cm_199607.tgz -> 31 días
Registros acumulados: 213
grids_germany_daily_soil_temperature_5cm_199608.tgz -> 31 días
Registros acumulados: 244
grids_germany_daily_soil_temperature_5cm_199609.tgz -> 30 días
Registros acumulados: 274
grids_germany_daily_soil_temperature_5cm_199610.tgz -> 31 días
Registros acumulados: 305
grids_germany_daily_soil_temperature_5cm_199611.tgz -> 30 día

grids_germany_daily_soil_temperature_5cm_200308.tgz -> 31 días
Registros acumulados: 2800
grids_germany_daily_soil_temperature_5cm_200309.tgz -> 30 días
Registros acumulados: 2830
grids_germany_daily_soil_temperature_5cm_200310.tgz -> 31 días
Registros acumulados: 2861
grids_germany_daily_soil_temperature_5cm_200311.tgz -> 30 días
Registros acumulados: 2891
grids_germany_daily_soil_temperature_5cm_200312.tgz -> 31 días
Registros acumulados: 2922
grids_germany_daily_soil_temperature_5cm_200401.tgz -> 31 días
Registros acumulados: 2953
grids_germany_daily_soil_temperature_5cm_200402.tgz -> 29 días
Registros acumulados: 2982
grids_germany_daily_soil_temperature_5cm_200403.tgz -> 31 días
Registros acumulados: 3013
grids_germany_daily_soil_temperature_5cm_200404.tgz -> 30 días
Registros acumulados: 3043
grids_germany_daily_soil_temperature_5cm_200405.tgz -> 31 días
Registros acumulados: 3074
grids_germany_daily_soil_temperature_5cm_200406.tgz -> 30 días
Registros acumulados: 3104
grids_germ

Registros acumulados: 5569
grids_germany_daily_soil_temperature_5cm_201104.tgz -> 30 días
Registros acumulados: 5599
grids_germany_daily_soil_temperature_5cm_201105.tgz -> 31 días
Registros acumulados: 5630
grids_germany_daily_soil_temperature_5cm_201106.tgz -> 30 días
Registros acumulados: 5660
grids_germany_daily_soil_temperature_5cm_201107.tgz -> 31 días
Registros acumulados: 5691
grids_germany_daily_soil_temperature_5cm_201108.tgz -> 31 días
Registros acumulados: 5722
grids_germany_daily_soil_temperature_5cm_201109.tgz -> 30 días
Registros acumulados: 5752
grids_germany_daily_soil_temperature_5cm_201110.tgz -> 31 días
Registros acumulados: 5783
grids_germany_daily_soil_temperature_5cm_201111.tgz -> 30 días
Registros acumulados: 5813
grids_germany_daily_soil_temperature_5cm_201112.tgz -> 31 días
Registros acumulados: 5844
grids_germany_daily_soil_temperature_5cm_201201.tgz -> 31 días
Registros acumulados: 5875
grids_germany_daily_soil_temperature_5cm_201202.tgz -> 29 días
Registros 

grids_germany_daily_soil_temperature_5cm_201811.tgz -> 30 días
Registros acumulados: 8370
grids_germany_daily_soil_temperature_5cm_201812.tgz -> 31 días
Registros acumulados: 8401
grids_germany_daily_soil_temperature_5cm_201901.tgz -> 31 días
Registros acumulados: 8432
grids_germany_daily_soil_temperature_5cm_201902.tgz -> 28 días
Registros acumulados: 8460
grids_germany_daily_soil_temperature_5cm_201903.tgz -> 31 días
Registros acumulados: 8491
grids_germany_daily_soil_temperature_5cm_201904.tgz -> 30 días
Registros acumulados: 8521
grids_germany_daily_soil_temperature_5cm_201905.tgz -> 31 días
Registros acumulados: 8552
grids_germany_daily_soil_temperature_5cm_201906.tgz -> 30 días
Registros acumulados: 8582
grids_germany_daily_soil_temperature_5cm_201907.tgz -> 31 días
Registros acumulados: 8613
grids_germany_daily_soil_temperature_5cm_201908.tgz -> 31 días
Registros acumulados: 8644
grids_germany_daily_soil_temperature_5cm_201909.tgz -> 30 días
Registros acumulados: 8674
grids_germ

In [4]:
df = pd.read_parquet('Sachsen_soil_temperature_5cm.parquet')
df.to_csv('Sachsen_soil_temperature_5cm.csv', index=False)

In [1]:
import pyarrow
print(pyarrow.__version__)

25.0.0


In [2]:
pip install --upgrade pyarrow

  Obtaining dependency information for pyarrow from https://files.pythonhosted.org/packages/f1/e2/738071e95c5ddad7b3dfc12f569ffa992db89d7d7b4a95258fd184191249/pyarrow-25.0.0-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/27.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.8 MB 320.0 kB/s eta 0:01:27
   ---------------------------------------- 0.0/27.8 MB 326.8 kB/s eta 0:01:26
   ---------------------------------------- 0.1/27.8 MB 435.7 kB/s eta 0:01:04
   ---------------------------------------- 0.2/27.8 MB 1.1 MB/s eta 0:00:25
    --------------------------------------- 0.5/27.8 MB 2.0 MB/s eta 0:00:14
    --------------------------------------- 0.6/27.8 MB 2.3 MB/s eta 0:00:13
   - -------------------------------------- 1.0/27.8 MB 3.4 MB/s eta 0:00:08
   - -------------------------------------- 1.0/27.8 MB 2.6 MB/s eta 0:00:11
   -- -----------------------

# Humedad del suelo (maíz)

In [4]:
"""
DWD Soil Moisture (Maize 0-10 cm) -> Daily mean over selected German states.

Adapted from HYRAS script.
"""
import re
from pathlib import Path
import geopandas as gpd
import pandas as pd
import requests
import rioxarray
import xarray as xr
from bs4 import BeautifulSoup
from tqdm import tqdm

BASE_ROOT = "https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/soil_moisture/maize/"
DEPTH="0-10"
AÑO_INICIO=1996
AÑO_FIN=2025
CARPETA_DESCARGA=Path("soil_moisture_nc")
GPKG_ESTADOS="States.gpkg"
CAMPO_NOMBRE_ESTADO="NUTS_NAME"
ESTADOS_INTERES=["Sachsen"]
SALIDA_CSV="Sachsen_SoilMoisture_0_10.csv"

def buscar_archivo(year):
    url=BASE_ROOT+f"{year}/"
    r=requests.get(url,timeout=30); r.raise_for_status()
    soup=BeautifulSoup(r.text,"html.parser")
    pat=re.compile(rf"grids_germany_daily_soil_moisture_maize_{year}_{DEPTH}_v(\d+)\.nc$")
    best=None
    for a in soup.find_all("a"):
        h=a.get("href","")
        m=pat.search(h)
        if m:
            v=int(m.group(1))
            if best is None or v>best[0]:
                best=(v,h)
    return None if best is None else url+best[1], None if best is None else best[1]

def descargar(url,nombre):
    CARPETA_DESCARGA.mkdir(exist_ok=True)
    dst=CARPETA_DESCARGA/nombre
    if dst.exists(): return dst
    with requests.get(url,stream=True,timeout=180) as r:
        r.raise_for_status()
        with open(dst,"wb") as f:
            for c in r.iter_content(1024*1024):
                f.write(c)
    return dst

rutas=[]
for y in tqdm(range(AÑO_INICIO,AÑO_FIN+1),desc="Descargando"):
    res=buscar_archivo(y)
    if res[0]:
        rutas.append(descargar(*res))
ds=xr.open_mfdataset(rutas,combine="by_coords")[["paws"]]
print(ds["paws"].attrs)
if ds.rio.crs is None:
    ds=ds.rio.write_crs("EPSG:31467",inplace=False)
est=gpd.read_file(GPKG_ESTADOS)
est=est[est[CAMPO_NOMBRE_ESTADO].isin(ESTADOS_INTERES)]
est=est.to_crs(ds.rio.crs)
geom=est.dissolve().geometry
clip=ds.rio.clip(geom,est.crs)
paws=clip["paws"].squeeze("lyr")
dims=[d for d in paws.dims if d!="time"]
serie=paws.mean(dim=dims,skipna=True)
df=serie.to_dataframe().reset_index().rename(columns={"time":"date","paws":"soil_moisture_0_10_%"})
df["year"]=df["date"].dt.year
df["month"]=df["date"].dt.month
df["day"]=df["date"].dt.day
df["doy"]=df["date"].dt.dayofyear
df=df[["date","year","month","day","doy","soil_moisture_0_10_%"]]
df.to_csv(SALIDA_CSV,index=False)
print(df.head())
print("Guardado:",SALIDA_CSV)

Descargando: 100%|██████████| 30/30 [00:53<00:00,  1.79s/it]


{'standard_name': 'percentage_of_plant_available_water_content_of_soil_layer', 'long_name': 'Percentage of plant available water content of soil layer', 'units': '% nFK', 'grid_mapping': 'transverse_mercator', 'cell_methods': 'time: point'}
        date  year  month  day  doy  soil_moisture_0_10_%
0 1996-01-01  1996      1    1    1            108.569052
1 1996-01-02  1996      1    2    2            107.591724
2 1996-01-03  1996      1    3    3            107.521070
3 1996-01-04  1996      1    4    4            106.737390
4 1996-01-05  1996      1    5    5            105.370580
Guardado: Sachsen_SoilMoisture_0_10.csv
